In [ ]:
%sql
-- Databricks SQL script: Comprehensive test suite for Excel-driven upsert into metric_config and metric_master tables
-- Purpose: Validate upsert logic, schema, data types, error handling, and data quality for metric_config and metric_master
-- Author: Giang Nguyen
-- Date: 2025-09-29
-- Description: This script tests upsert operations from Excel staging tables into purgo_playground.metric_config and purgo_playground.metric_master, including schema validation, data type checks, NULL handling, duplicate detection, and success confirmation. It uses CTEs for test data, validation queries, and asserts correct behavior for all edge cases.

USE CATALOG purgo_databricks;

-- =========================
-- CTE: Test Data for metric_config staging
-- Covers happy path, edge cases, error cases, NULLs, and special characters
-- =========================
WITH metric_config_stg AS (
  SELECT "evenity_unit_hash" AS metric_id, "Y" AS active_indicator, NULL AS geographical_average_type, "String" AS metric_data_type,
         "hash value of product level , product name and market " AS metric_description, "evenity_unit_hash" AS metric_template_name,
         "Custom" AS metric_type, NULL AS optional_filters, "purgo_playground.s_field_reporting_sales_source_customer" AS source_table,
         "customer" AS table_type, "purgo_playground.s_field_reporting_sales_source_customer_metric" AS target_table,
         NULL AS template_parameters, NULL AS bu_filter, NULL AS calling_service_name, "pt_cdl_uuid" AS primary_key
  UNION ALL
  SELECT "customer_first_metric", "Y", NULL, NULL,
         "First purchased columns of a customer", "customer_first_metric", "Custom", NULL,
         "purgo_playground.d_product_revenue", "Product", "purgo_playground.d_product_revenue_metric",
         NULL, NULL, NULL, "product_id"
  UNION ALL
  SELECT "final_css", "Y", NULL, "Double",
         "Normalised Customer Satisfaction Score", "final_css", "Custom", NULL,
         "purgo_playground.health_insurance_claims", "HCP", "purgo_playground.health_insurance_claims_metric",
         NULL, NULL, NULL, "Claim_ID"
  UNION ALL
  SELECT "batch_number", "N", NULL, "String",
         "Batch number of product", "batch_number", "Custom", NULL,
         "purgo_playground.d_product_revenue", "Product", "purgo_playground.d_product_revenue_metric",
         NULL, NULL, NULL, "product_id"
  -- Edge case: NULL metric_id (should fail validation)
  UNION ALL
  SELECT NULL, "Y", NULL, "String",
         "Missing metric_id", "missing_metric_id", "Custom", NULL,
         "purgo_playground.d_product_revenue", "Product", "purgo_playground.d_product_revenue_metric",
         NULL, NULL, NULL, "product_id"
  -- Error case: Invalid metric_data_type (numeric instead of string)
  UNION ALL
  SELECT "final_css_invalid_type", "Y", NULL, "123",
         "Invalid metric_data_type", "final_css", "Custom", NULL,
         "purgo_playground.health_insurance_claims", "HCP", "purgo_playground.health_insurance_claims_metric",
         NULL, NULL, NULL, "Claim_ID"
  -- Error case: Invalid active_indicator (numeric instead of string)
  UNION ALL
  SELECT "batch_number_invalid_active", "5", NULL, "String",
         "Invalid active_indicator", "batch_number", "Custom", NULL,
         "purgo_playground.d_product_revenue", "Product", "purgo_playground.d_product_revenue_metric",
         NULL, NULL, NULL, "product_id"
  -- Edge case: Duplicate metric_id in staging (should fail validation)
  UNION ALL
  SELECT "evenity_unit_hash", "Y", NULL, "String",
         "Duplicate metric_id", "evenity_unit_hash", "Custom", NULL,
         "purgo_playground.s_field_reporting_sales_source_customer", "customer", "purgo_playground.s_field_reporting_sales_source_customer_metric",
         NULL, NULL, NULL, "pt_cdl_uuid"
  -- NULL handling: All fields NULL except metric_id
  UNION ALL
  SELECT "null_fields_metric", NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL
  -- Special characters: metric_id with emoji and multi-byte chars
  UNION ALL
  SELECT "batch_№_测试_🚀", "Y", NULL, "String",
         "Batch number with special chars: №, 测试, 🚀", "batch_number_special", "Custom", NULL,
         "purgo_playground.d_product_revenue", "Product", "purgo_playground.d_product_revenue_metric",
         NULL, NULL, NULL, "product_id"
  -- Edge case: Overly long metric_id
  UNION ALL
  SELECT RPAD("long_metric_id_", 100, "X"), "Y", NULL, "String",
         "Overly long metric_id", "long_metric_id", "Custom", NULL,
         "purgo_playground.d_product_revenue", "Product", "purgo_playground.d_product_revenue_metric",
         NULL, NULL, NULL, "product_id"
  -- Error case: Missing required metric_template_name
  UNION ALL
  SELECT "missing_template_name", "Y", NULL, "String",
         "Missing metric_template_name", NULL, "Custom", NULL,
         "purgo_playground.d_product_revenue", "Product", "purgo_playground.d_product_revenue_metric",
         NULL, NULL, NULL, "product_id"
  -- Error case: Missing required metric_type
  UNION ALL
  SELECT "missing_metric_type", "Y", NULL, "String",
         "Missing metric_type", "missing_metric_type", NULL, NULL,
         "purgo_playground.d_product_revenue", "Product", "purgo_playground.d_product_revenue_metric",
         NULL, NULL, NULL, "product_id"
  -- Happy path: All fields populated
  UNION ALL
  SELECT "full_fields_metric", "Y", "AVG", "Decimal(10,2)",
         "All fields populated", "full_fields_metric", "Custom", "filter1,filter2",
         "purgo_playground.d_product_revenue", "Product", "purgo_playground.d_product_revenue_metric",
         "param1,param2", "BU1", "service1", "product_id"
  -- Edge case: NULL primary_key
  UNION ALL
  SELECT "null_primary_key", "Y", NULL, "String",
         "NULL primary_key", "null_primary_key", "Custom", NULL,
         "purgo_playground.d_product_revenue", "Product", "purgo_playground.d_product_revenue_metric",
         NULL, NULL, NULL, NULL
  -- Error case: metric_id as integer (should fail type validation)
  UNION ALL
  SELECT CAST(123 AS STRING), "Y", NULL, "String",
         "metric_id as integer", "int_metric_id", "Custom", NULL,
         "purgo_playground.d_product_revenue", "Product", "purgo_playground.d_product_revenue_metric",
         NULL, NULL, NULL, "product_id"
  -- Edge case: Empty string metric_id
  UNION ALL
  SELECT "", "Y", NULL, "String",
         "Empty string metric_id", "empty_metric_id", "Custom", NULL,
         "purgo_playground.d_product_revenue", "Product", "purgo_playground.d_product_revenue_metric",
         NULL, NULL, NULL, "product_id"
  -- Happy path: Special characters in description
  UNION ALL
  SELECT "special_desc_metric", "Y", NULL, "String",
         "Description with special chars: !@#$%^&*()_+[]{}|;:,.<>/?", "special_desc_metric", "Custom", NULL,
         "purgo_playground.d_product_revenue", "Product", "purgo_playground.d_product_revenue_metric",
         NULL, NULL, NULL, "product_id"
  -- Edge case: NULL metric_description
  UNION ALL
  SELECT "null_desc_metric", "Y", NULL, "String",
         NULL, "null_desc_metric", "Custom", NULL,
         "purgo_playground.d_product_revenue", "Product", "purgo_playground.d_product_revenue_metric",
         NULL, NULL, NULL, "product_id"
  -- Error case: metric_id with only spaces
  UNION ALL
  SELECT "   ", "Y", NULL, "String",
         "metric_id with only spaces", "spaces_metric_id", "Custom", NULL,
         "purgo_playground.d_product_revenue", "Product", "purgo_playground.d_product_revenue_metric",
         NULL, NULL, NULL, "product_id"
)

-- =========================
-- CTE: Test Data for metric_master staging
-- Covers happy path, edge cases, error cases, NULLs, and special characters
-- =========================
, metric_master_stg AS (
  SELECT "evenity_unit_hash" AS metric_template_name,
         "SELECT * FROM (SELECT source_customer.*, time_bucket.time_bucket_start_date, time_bucket.time_bucket_end_date FROM agilisium_playground.purgo_playground.s_field_reporting_sales_source_customer source_customer INNER JOIN agilisium_playground.purgo_playground.s_field_reporting_sales_time_group_bucket time_bucket ON source_customer.pt_cdl_uuid = time_bucket.pt_cdl_uuid AND LOWER(source_customer.time_bucket_id) = LOWER(time_bucket.time_bucket_id) AND LOWER(source_customer.cdl_frequency) = LOWER(time_bucket.cdl_frequency) AND LOWER(source_customer.field_force_code) = LOWER(time_bucket.field_force_code) WHERE product_name = ""EVENITY"" AND product_level = ""BRAND"" AND market_name = ""PMO TOTAL - BHBU"")" AS sql_query,
         "Custom" AS metric_type, "eve_view" AS view_name, 1 AS dependency
  UNION ALL
  SELECT "customer_first_metric",
         "SELECT product_id, FIRST_VALUE(purchased_date) OVER (PARTITION BY customer_id ORDER BY purchased_date) AS first_purchase_date, FIRST_VALUE(product_id) OVER (PARTITION BY customer_id ORDER BY purchased_date) AS first_purchase_product, FIRST_VALUE(revenue) OVER (PARTITION BY customer_id ORDER BY purchased_date) AS first_revenue FROM purgo_playground.d_product_revenue;",
         "Custom", NULL, 1
  UNION ALL
  SELECT "final_css",
         "SELECT *, CASE WHEN Billed_Amount = 0 THEN 0 ELSE (0.2 * (Allowed_Amount / Billed_Amount) * 100) - (0.3 * (Patient_Paid / Billed_Amount) * 100) - (0.2 * DATEDIFF(Service_Date, service_requested_date)) + (0.2 * SIZE(SPLIT(purchase_history, ','))) - (0.1 * (1 - is_churn)) END AS CSS FROM joined_data;",
         "Custom", "css_data", 2
  UNION ALL
  SELECT "final_css",
         "WITH css_stats AS (SELECT MIN(CSS) AS min_css, MAX(CSS) AS max_css FROM css_data) SELECT Claim_ID,CASE WHEN max_css = min_css THEN 0 ELSE ((CSS - min_css) / (max_css - min_css)) * 10 END AS final_CSS FROM css_data, css_stats;",
         "Custom", NULL, 3
  UNION ALL
  SELECT "final_css",
         "SELECT h.*, c.id, c.purchase_history, c.is_churn FROM purgo_playground.health_insurance_claims h JOIN purgo_playground.customer_360_raw_with_churn c ON h.Patient_ID = c.id;",
         "Custom", "joined_data", 1
  UNION ALL
  SELECT "evenity_unit_hash",
         "SELECT pt_cdl_uuid, HASH(CONCAT_WS('_','PGP',market_name,product_level,product_name)) AS evenity_unit_hash FROM eve_view",
         "Custom", NULL, 2
  UNION ALL
  SELECT "batch_number",
         "SELECT product_id, get_json_object(product_details, '$.batch_number') AS batch_number FROM purgo_playground.d_product_revenue",
         "Custom", NULL, 1
  -- Edge case: NULL metric_template_name (should fail validation)
  UNION ALL
  SELECT NULL, "SELECT 1", "Custom", NULL, 1
  -- Error case: Invalid dependency (string instead of tinyint)
  UNION ALL
  SELECT "final_css", "SELECT 2", "Custom", NULL, CAST("abc" AS STRING)
  -- Error case: Invalid metric_type (numeric instead of string)
  UNION ALL
  SELECT "batch_number", "SELECT 3", CAST(123 AS STRING), NULL, 1
  -- Edge case: Duplicate (metric_template_name, dependency) in staging (should fail validation)
  UNION ALL
  SELECT "final_css", "SELECT duplicate", "Custom", NULL, 2
  -- NULL handling: All fields NULL except metric_template_name
  UNION ALL
  SELECT "null_fields_master", NULL, NULL, NULL, NULL
  -- Special characters: metric_template_name with emoji and multi-byte chars
  UNION ALL
  SELECT "batch_№_测试_🚀", "SELECT special chars", "Custom", NULL, 1
  -- Edge case: Overly long metric_template_name
  UNION ALL
  SELECT RPAD("long_template_name_", 100, "X"), "SELECT long name", "Custom", NULL, 1
  -- Error case: Missing required sql_query
  UNION ALL
  SELECT "missing_sql_query", NULL, "Custom", NULL, 2
  -- Error case: Missing required metric_type
  UNION ALL
  SELECT "missing_metric_type", "SELECT missing type", NULL, NULL, 1
  -- Happy path: All fields populated
  UNION ALL
  SELECT "full_fields_master", "SELECT * FROM full_fields", "Custom", "full_view", 1
  -- Edge case: NULL dependency
  UNION ALL
  SELECT "null_dependency_master", "SELECT * FROM null_dep", "Custom", NULL, NULL
  -- Error case: dependency as integer string (should fail type validation)
  UNION ALL
  SELECT "int_string_dependency", "SELECT * FROM int_string", "Custom", NULL, CAST("not_a_number" AS STRING)
  -- Edge case: Empty string metric_template_name
  UNION ALL
  SELECT "", "SELECT empty name", "Custom", NULL, 1
  -- Happy path: Special characters in sql_query
  UNION ALL
  SELECT "special_sql_query", "SELECT * FROM table WHERE col LIKE '!@#$%^&*()_+[]{}|;:,.<>/?'", "Custom", NULL, 1
  -- Edge case: NULL view_name
  UNION ALL
  SELECT "null_view_name_master", "SELECT * FROM null_view", "Custom", NULL, 1
  -- Error case: metric_template_name with only spaces
  UNION ALL
  SELECT "   ", "SELECT spaces", "Custom", NULL, 1
)

-- =========================
-- MERGE: Upsert into purgo_playground.metric_config
-- If metric_id exists, update; else insert
-- =========================
MERGE INTO purgo_databricks.purgo_playground.metric_config AS tgt
USING metric_config_stg AS src
ON tgt.metric_id = src.metric_id
WHEN MATCHED THEN
  UPDATE SET
    tgt.active_indicator = src.active_indicator,
    tgt.geographical_average_type = src.geographical_average_type,
    tgt.metric_data_type = src.metric_data_type,
    tgt.metric_description = src.metric_description,
    tgt.metric_template_name = src.metric_template_name,
    tgt.metric_type = src.metric_type,
    tgt.optional_filters = src.optional_filters,
    tgt.source_table = src.source_table,
    tgt.table_type = src.table_type,
    tgt.target_table = src.target_table,
    tgt.template_parameters = src.template_parameters,
    tgt.bu_filter = src.bu_filter,
    tgt.calling_service_name = src.calling_service_name,
    tgt.primary_key = src.primary_key
WHEN NOT MATCHED THEN
  INSERT (
    metric_id,
    active_indicator,
    geographical_average_type,
    metric_data_type,
    metric_description,
    metric_template_name,
    metric_type,
    optional_filters,
    source_table,
    table_type,
    target_table,
    template_parameters,
    bu_filter,
    calling_service_name,
    primary_key
  )
  VALUES (
    src.metric_id,
    src.active_indicator,
    src.geographical_average_type,
    src.metric_data_type,
    src.metric_description,
    src.metric_template_name,
    src.metric_type,
    src.optional_filters,
    src.source_table,
    src.table_type,
    src.target_table,
    src.template_parameters,
    src.bu_filter,
    src.calling_service_name,
    src.primary_key
  );

-- =========================
-- MERGE: Upsert into purgo_playground.metric_master
-- If (metric_template_name, dependency) exists, update; else insert
-- NULL-safe matching for dependency
-- =========================

WITH metric_config_stg AS (
  SELECT "evenity_unit_hash" AS metric_id, "Y" AS active_indicator, NULL AS geographical_average_type, "String" AS metric_data_type,
         "hash value of product level , product name and market " AS metric_description, "evenity_unit_hash" AS metric_template_name,
         "Custom" AS metric_type, NULL AS optional_filters, "purgo_playground.s_field_reporting_sales_source_customer" AS source_table,
         "customer" AS table_type, "purgo_playground.s_field_reporting_sales_source_customer_metric" AS target_table,
         NULL AS template_parameters, NULL AS bu_filter, NULL AS calling_service_name, "pt_cdl_uuid" AS primary_key
  UNION ALL
  SELECT "customer_first_metric", "Y", NULL, NULL,
         "First purchased columns of a customer", "customer_first_metric", "Custom", NULL,
         "purgo_playground.d_product_revenue", "Product", "purgo_playground.d_product_revenue_metric",
         NULL, NULL, NULL, "product_id"
  UNION ALL
  SELECT "final_css", "Y", NULL, "Double",
         "Normalised Customer Satisfaction Score", "final_css", "Custom", NULL,
         "purgo_playground.health_insurance_claims", "HCP", "purgo_playground.health_insurance_claims_metric",
         NULL, NULL, NULL, "Claim_ID"
  UNION ALL
  SELECT "batch_number", "N", NULL, "String",
         "Batch number of product", "batch_number", "Custom", NULL,
         "purgo_playground.d_product_revenue", "Product", "purgo_playground.d_product_revenue_metric",
         NULL, NULL, NULL, "product_id"
  UNION ALL
  SELECT NULL, "Y", NULL, "String",
         "Missing metric_id", "missing_metric_id", "Custom", NULL,
         "purgo_playground.d_product_revenue", "Product", "purgo_playground.d_product_revenue_metric",
         NULL, NULL, NULL, "product_id"
  UNION ALL
  SELECT "final_css_invalid_type", "Y", NULL, "123",
         "Invalid metric_data_type", "final_css", "Custom", NULL,
         "purgo_playground.health_insurance_claims", "HCP", "purgo_playground.health_insurance_claims_metric",
         NULL, NULL, NULL, "Claim_ID"
  UNION ALL
  SELECT "batch_number_invalid_active", "5", NULL, "String",
         "Invalid active_indicator", "batch_number", "Custom", NULL,
         "purgo_playground.d_product_revenue", "Product", "purgo_playground.d_product_revenue_metric",
         NULL, NULL, NULL, "product_id"
  UNION ALL
  SELECT "evenity_unit_hash", "Y", NULL, "String",
         "Duplicate metric_id", "evenity_unit_hash", "Custom", NULL,
         "purgo_playground.s_field_reporting_sales_source_customer", "customer", "purgo_playground.s_field_reporting_sales_source_customer_metric",
         NULL, NULL, NULL, "pt_cdl_uuid"
  UNION ALL
  SELECT "null_fields_metric", NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL
  UNION ALL
  SELECT "batch_№_测试_🚀", "Y", NULL, "String",
         "Batch number with special chars: №, 测试, 🚀", "batch_number_special", "Custom", NULL,
         "purgo_playground.d_product_revenue", "Product", "purgo_playground.d_product_revenue_metric",
         NULL, NULL, NULL, "product_id"
  UNION ALL
  SELECT RPAD("long_metric_id_", 100, "X"), "Y", NULL, "String",
         "Overly long metric_id", "long_metric_id", "Custom", NULL,
         "purgo_playground.d_product_revenue", "Product", "purgo_playground.d_product_revenue_metric",
         NULL, NULL, NULL, "product_id"
  UNION ALL
  SELECT "missing_template_name", "Y", NULL, "String",
         "Missing metric_template_name", NULL, "Custom", NULL,
         "purgo_playground.d_product_revenue", "Product", "purgo_playground.d_product_revenue_metric",
         NULL, NULL, NULL, "product_id"
  UNION ALL
  SELECT "missing_metric_type", "Y", NULL, "String",
         "Missing metric_type", "missing_metric_type", NULL, NULL,
         "purgo_playground.d_product_revenue", "Product", "purgo_playground.d_product_revenue_metric",
         NULL, NULL, NULL, "product_id"
  UNION ALL
  SELECT "full_fields_metric", "Y", "AVG", "Decimal(10,2)",
         "All fields populated", "full_fields_metric", "Custom", "filter1,filter2",
         "purgo_playground.d_product_revenue", "Product", "purgo_playground.d_product_revenue_metric",
         "param1,param2", "BU1", "service1", "product_id"
  UNION ALL
  SELECT "null_primary_key", "Y", NULL, "String",
         "NULL primary_key", "null_primary_key", "Custom", NULL,
         "purgo_playground.d_product_revenue", "Product", "purgo_playground.d_product_revenue_metric",
         NULL, NULL, NULL, NULL
  UNION ALL
  SELECT CAST(123 AS STRING), "Y", NULL, "String",
         "metric_id as integer", "int_metric_id", "Custom", NULL,
         "purgo_playground.d_product_revenue", "Product", "purgo_playground.d_product_revenue_metric",
         NULL, NULL, NULL, "product_id"
  UNION ALL
  SELECT "", "Y", NULL, "String",
         "Empty string metric_id", "empty_metric_id", "Custom", NULL,
         "purgo_playground.d_product_revenue", "Product", "purgo_playground.d_product_revenue_metric",
         NULL, NULL, NULL, "product_id"
  UNION ALL
  SELECT "special_desc_metric", "Y", NULL, "String",
         "Description with special chars: !@#$%^&*()_+[]{}|;:,.<>/?", "special_desc_metric", "Custom", NULL,
         "purgo_playground.d_product_revenue", "Product", "purgo_playground.d_product_revenue_metric",
         NULL, NULL, NULL, "product_id"
  UNION ALL
  SELECT "null_desc_metric", "Y", NULL, "String",
         NULL, "null_desc_metric", "Custom", NULL,
         "purgo_playground.d_product_revenue", "Product", "purgo_playground.d_product_revenue_metric",
         NULL, NULL, NULL, "product_id"
  UNION ALL
  SELECT "   ", "Y", NULL, "String",
         "metric_id with only spaces", "spaces_metric_id", "Custom", NULL,
         "purgo_playground.d_product_revenue", "Product", "purgo_playground.d_product_revenue_metric",
         NULL, NULL, NULL, "product_id"
)
, metric_master_stg AS (
  SELECT "evenity_unit_hash" AS metric_template_name,
         "SELECT * FROM (SELECT source_customer.*, time_bucket.time_bucket_start_date, time_bucket.time_bucket_end_date FROM agilisium_playground.purgo_playground.s_field_reporting_sales_source_customer source_customer INNER JOIN agilisium_playground.purgo_playground.s_field_reporting_sales_time_group_bucket time_bucket ON source_customer.pt_cdl_uuid = time_bucket.pt_cdl_uuid AND LOWER(source_customer.time_bucket_id) = LOWER(time_bucket.time_bucket_id) AND LOWER(source_customer.cdl_frequency) = LOWER(time_bucket.cdl_frequency) AND LOWER(source_customer.field_force_code) = LOWER(time_bucket.field_force_code) WHERE product_name = ""EVENITY"" AND product_level = ""BRAND"" AND market_name = ""PMO TOTAL - BHBU"")" AS sql_query,
         "Custom" AS metric_type, "eve_view" AS view_name, 1 AS dependency
  UNION ALL
  SELECT "customer_first_metric",
         "SELECT product_id, FIRST_VALUE(purchased_date) OVER (PARTITION BY customer_id ORDER BY purchased_date) AS first_purchase_date, FIRST_VALUE(product_id) OVER (PARTITION BY customer_id ORDER BY purchased_date) AS first_purchase_product, FIRST_VALUE(revenue) OVER (PARTITION BY customer_id ORDER BY purchased_date) AS first_revenue FROM purgo_playground.d_product_revenue;",
         "Custom", NULL, 1
  UNION ALL
  SELECT "final_css",
         "SELECT *, CASE WHEN Billed_Amount = 0 THEN 0 ELSE (0.2 * (Allowed_Amount / Billed_Amount) * 100) - (0.3 * (Patient_Paid / Billed_Amount) * 100) - (0.2 * DATEDIFF(Service_Date, service_requested_date)) + (0.2 * SIZE(SPLIT(purchase_history, ','))) - (0.1 * (1 - is_churn)) END AS CSS FROM joined_data;",
         "Custom", "css_data", 2
  UNION ALL
  SELECT "final_css",
         "WITH css_stats AS (SELECT MIN(CSS) AS min_css, MAX(CSS) AS max_css FROM css_data) SELECT Claim_ID,CASE WHEN max_css = min_css THEN 0 ELSE ((CSS - min_css) / (max_css - min_css)) * 10 END AS final_CSS FROM css_data, css_stats;",
         "Custom", NULL, 3
  UNION ALL
  SELECT "final_css",
         "SELECT h.*, c.id, c.purchase_history, c.is_churn FROM purgo_playground.health_insurance_claims h JOIN purgo_playground.customer_360_raw_with_churn c ON h.Patient_ID = c.id;",
         "Custom", "joined_data", 1
  UNION ALL
  SELECT "evenity_unit_hash",
         "SELECT pt_cdl_uuid, HASH(CONCAT_WS('_','PGP',market_name,product_level,product_name)) AS evenity_unit_hash FROM eve_view",
         "Custom", NULL, 2
  UNION ALL
  SELECT "batch_number",
         "SELECT product_id, get_json_object(product_details, '$.batch_number') AS batch_number FROM purgo_playground.d_product_revenue",
         "Custom", NULL, 1
  UNION ALL
  SELECT NULL, "SELECT 1", "Custom", NULL, 1
  UNION ALL
  SELECT "final_css", "SELECT 2", "Custom", NULL, CAST("abc" AS STRING)
  UNION ALL
  SELECT "batch_number", "SELECT 3", CAST(123 AS STRING), NULL, 1
  UNION ALL
  SELECT "final_css", "SELECT duplicate", "Custom", NULL, 2
  UNION ALL
  SELECT "null_fields_master", NULL, NULL, NULL, NULL
  UNION ALL
  SELECT "batch_№_测试_🚀", "SELECT special chars", "Custom", NULL, 1
  UNION ALL
  SELECT RPAD("long_template_name_", 100, "X"), "SELECT long name", "Custom", NULL, 1
  UNION ALL
  SELECT "missing_sql_query", NULL, "Custom", NULL, 2
  UNION ALL
  SELECT "missing_metric_type", "SELECT missing type", NULL, NULL, 1
  UNION ALL
  SELECT "full_fields_master", "SELECT * FROM full_fields", "Custom", "full_view", 1
  UNION ALL
  SELECT "null_dependency_master", "SELECT * FROM null_dep", "Custom", NULL, NULL
  UNION ALL
  SELECT "int_string_dependency", "SELECT * FROM int_string", "Custom", NULL, CAST("not_a_number" AS STRING)
  UNION ALL
  SELECT "", "SELECT empty name", "Custom", NULL, 1
  UNION ALL
  SELECT "special_sql_query", "SELECT * FROM table WHERE col LIKE '!@#$%^&*()_+[]{}|;:,.<>/?'", "Custom", NULL, 1
  UNION ALL
  SELECT "null_view_name_master", "SELECT * FROM null_view", "Custom", NULL, 1
  UNION ALL
  SELECT "   ", "SELECT spaces", "Custom", NULL, 1
)

MERGE INTO purgo_databricks.purgo_playground.metric_master AS tgt
USING metric_master_stg AS src
ON tgt.metric_template_name <=> src.metric_template_name AND tgt.dependency <=> src.dependency
WHEN MATCHED THEN
  UPDATE SET
    tgt.sql_query = src.sql_query,
    tgt.metric_type = src.metric_type,
    tgt.view_name = src.view_name
WHEN NOT MATCHED THEN
  INSERT (
    metric_template_name,
    sql_query,
    metric_type,
    view_name,
    dependency
  )
  VALUES (
    src.metric_template_name,
    src.sql_query,
    src.metric_type,
    src.view_name,
    src.dependency
  );

-- =========================
-- CTE: Validation Query for metric_config
-- Checks for duplicate metric_id, missing required fields, and invalid data types
-- =========================
WITH metric_config_validation AS (
  SELECT
    metric_id,
    COUNT(*) AS cnt,
    CASE
      WHEN metric_id IS NULL OR TRIM(metric_id) = "" THEN "Missing required field: metric_id"
      WHEN metric_template_name IS NULL OR TRIM(metric_template_name) = "" THEN "Missing required field: metric_template_name"
      WHEN metric_type IS NULL OR TRIM(metric_type) = "" THEN "Missing required field: metric_type"
      WHEN TRY_CAST(metric_data_type AS STRING) IS NULL AND metric_data_type IS NOT NULL THEN CONCAT("Invalid type for metric_data_type: ", metric_data_type)
      WHEN TRY_CAST(active_indicator AS STRING) IS NULL AND active_indicator IS NOT NULL THEN CONCAT("Invalid type for active_indicator: ", active_indicator)
      ELSE NULL
    END AS error_message
  FROM metric_config_stg
  GROUP BY metric_id, metric_template_name, metric_type, metric_data_type, active_indicator
)
SELECT * FROM metric_config_validation WHERE cnt > 1 OR error_message IS NOT NULL;

-- =========================
-- CTE: Validation Query for metric_master
-- Checks for duplicate (metric_template_name, dependency), missing required fields, and invalid data types
-- =========================
WITH metric_master_validation AS (
  SELECT
    metric_template_name,
    dependency,
    COUNT(*) AS cnt,
    CASE
      WHEN metric_template_name IS NULL OR TRIM(metric_template_name) = "" THEN "Missing required field: metric_template_name"
      WHEN sql_query IS NULL OR TRIM(sql_query) = "" THEN "Missing required field: sql_query"
      WHEN metric_type IS NULL OR TRIM(metric_type) = "" THEN "Missing required field: metric_type"
      WHEN TRY_CAST(dependency AS TINYINT) IS NULL AND dependency IS NOT NULL THEN CONCAT("Invalid type for dependency: ", dependency)
      ELSE NULL
    END AS error_message
  FROM metric_master_stg
  GROUP BY metric_template_name, dependency, sql_query, metric_type
)
SELECT * FROM metric_master_validation WHERE cnt > 1 OR error_message IS NOT NULL;

-- =========================
-- Success confirmation: Validate upserted data matches Excel data
-- =========================
-- Validate metric_config upsert
SELECT
  tgt.metric_id,
  tgt.active_indicator,
  tgt.metric_data_type,
  tgt.metric_description,
  tgt.metric_template_name,
  tgt.metric_type,
  tgt.source_table,
  tgt.table_type,
  tgt.target_table,
  tgt.primary_key
FROM purgo_databricks.purgo_playground.metric_config tgt
WHERE tgt.metric_id IN (
  "evenity_unit_hash", "customer_first_metric", "final_css", "batch_number", "full_fields_metric", "batch_№_测试_🚀"
);

-- Validate metric_master upsert
SELECT
  tgt.metric_template_name,
  tgt.sql_query,
  tgt.metric_type,
  tgt.view_name,
  tgt.dependency
FROM purgo_databricks.purgo_playground.metric_master tgt
WHERE (tgt.metric_template_name, tgt.dependency) IN (
  ("final_css", 2), ("evenity_unit_hash", 1), ("customer_first_metric", 1), ("batch_number", 1), ("full_fields_master", 1), ("batch_№_测试_🚀", 1)
);

-- =========================
-- Cleanup: No temp views or temp tables used, no cleanup required
-- =========================
